# Introduction

In this work, we embark on an exciting investigation into predictive technologies, with the aim of creating sophisticated models that can predict age and sex from facial expressions. Our journey leads to the convergence of advanced deep learning techniques, specifically Convolutional Neural Networks (CNNs), using the UTKFace database. The main objective of this project is to highlight the complexity of multitasking learning and, at the same time, to meet the challenging challenge of age-related prediction of sex prediction time same by other computer methods.

## Purpose:

At the heart of this work lies the main goal – to develop and implement a sophisticated CNN model that excels age and sex prediction simultaneously from facial images. By combining these two predictive functions, we aim to highlight the potential of multitask learning in real-world applications. This effort utilizes the synchronous synergies between advanced CNN algorithms, the extensive UTKFace dataset, and the methodological principles that guide model design The ultimate goal is to uncover mysteries which has predictive multivariate outcomes in the specific context of age and sex estimates.

## Problem Statement:

Drawing robust predictions of age and sex from facial characteristics is a central challenge of our university project. This challenging terrain calls for the development of new CNN algorithms that can accommodate multidimensional age progression and gender distributions Our effort is primarily to create a model that does not necessarily give age and gender not only accurate sexual prediction but effectively manages the complexities of these separate but interacting processes results.

## Defining and selecting a data set:

In this exercise, we strategically chose the UTKFace data set as our cornerstone. This dataset is a rich repository of face images across a wide range of ages, genders, and ethnicities. With over 20,000 carefully labeled images in the dataset, it stands as a valuable cornerstone for training, optimization, and evaluation of our model. Including age, gender, and ethnicity information in the dataset ensures solid ground truth for our ambitious multidisciplinary study efforts.

Our decision to select the UTKFace dataset is based on its diversity, which reflects the real-world complexities of our model. The broad spectrum of age and sex in the data sets allows us to elucidate the complexity of age and key characteristics associated with sex. Furthermore, the large size of the dataset enables our deep CNN model to identify complex facial features and their relationships.

By taking advantage of the diverse features in the UTKFace dataset, we aim to demonstrate the robustness of the model across different age, gender, and ethnic groups. This data set fits well with our project’s main objective of relevance and accuracy under practical conditions. Through extensive testing and analysis, we intend to push the limits of the predictive capabilities of our model and verify its generalizability.

# Methodology

## Data Loading and Prepocessing

In [ ]:
from google.colab import drive #connecting to drive
drive.mount('/content/gdrive')

In [ ]:
!unzip /content/gdrive/MyDrive/data_for_mtl.zip -d data #unzipping data

### Loading Packages

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import cv2
from keras.models import Sequential,load_model,Model
from keras.layers import Conv2D,MaxPool2D,Dense,Dropout,BatchNormalization,Flatten,Input, MaxPooling2D, concatenate
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from tensorflow.keras.preprocessing.image import load_img
from tqdm.notebook import tqdm
from tensorflow.keras.optimizers import Adam
from tqdm import tqdm
from tensorflow.keras import backend as K
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from keras.callbacks import Callback, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from tensorflow.keras.utils import plot_model
from PIL import Image
from sklearn.metrics import roc_curve, auc

### Extracting Labels from Dataset

In [ ]:
path = "/content/data/utkcropped/utkcropped"  # Directory path
image_paths = []
age_labels = []
gender_labels = []

for filename in tqdm(os.listdir(path)):
    if filename.endswith('.jpg'):  # Filter out non-image files
        try:
            components = filename[:-4].split('_')  # Remove ".jpg" and split
            age = int(components[0])
            gender = int(components[1])
            image_paths.append(os.path.join(path, filename))
            age_labels.append(age)
            gender_labels.append(gender)
        except ValueError:
            # This block will catch the ValueError for filenames that don't match the expected format
            print(f"Ignoring file: {filename}")


In [ ]:
df = pd.DataFrame() #converting labels into dataframe
df['image'], df['age'], df['gender'] = image_paths, age_labels, gender_labels
df.head()

It seem that all image's age and gender which are label has been extracted

In [ ]:
df.shape

In [ ]:
#alotting gender according to 0 and 1
gender_dict = {0:'Male', 1:'Female'}

In [ ]:
plt.figure(figsize=(20, 20))  #loading first 25 images with there age and gender
files = df.iloc[0:25]

for index, file, age, gender in files.itertuples():
    plt.subplot(5, 5, index+1)
    img = load_img(file)
    img = np.array(img)
    plt.imshow(img)
    plt.title(f"Age: {age} Gender: {gender_dict[gender]} ")
    plt.axis('off')

It can be seen that all images are has there respective gender and age.

### Extract feature from Images

In [ ]:
def extract_features(images):  #extraxting features from images
    features = []
    for image in tqdm(images):
        img = load_img(image, grayscale=True)
        img = img.resize((128,128), Image.ANTIALIAS)
        img = np.array(img)
        features.append(img)

    features = np.array(features)
    # ignore this step if using RGB
    features = features.reshape(len(features),128,128, 1) #reshape them into (128,128)
    return features

In [ ]:
X1 = extract_features(df['image']) #defining image features as our X1

In [ ]:
X1.shape

In [ ]:
X = X1/255.0

In [ ]:
y = np.array(df['gender'])#defining our label

In [ ]:
y2 = np.array(df['age']) #defining our second output of age

## Multi task Model

In [ ]:
#splitting data in training and testing
X_train, X_test, y_train, y_test, y2_train, y2_test = train_test_split(X, y, y2, test_size=0.2, random_state=42)

In [ ]:
# Custom R2 metric function
def r2_score(y_true, y_pred):
    SS_res = K.sum(K.square(y_true - y_pred))
    SS_tot = K.sum(K.square(y_true - K.mean(y_true)))
    r2 = 1 - SS_res / (SS_tot + K.epsilon())
    return r2

In [ ]:
inputs = Input(shape=(128, 128, 1))  # image input CNN model

# CNN branch
conv_1 = Conv2D(32, kernel_size=(3, 3), activation='relu')(inputs)
maxp_1 = MaxPooling2D(pool_size=(2, 2))(conv_1)
conv_2 = Conv2D(64, kernel_size=(3, 3), activation='relu')(maxp_1)
maxp_2 = MaxPooling2D(pool_size=(2, 2))(conv_2)
conv_3 = Conv2D(128, kernel_size=(3, 3), activation='relu')(maxp_2)
maxp_3 = MaxPooling2D(pool_size=(2, 2))(conv_3)
conv_4 = Conv2D(256, kernel_size=(3, 3), activation='relu')(maxp_3)
maxp_4 = MaxPooling2D(pool_size=(2, 2))(conv_4)
conv_5 = Conv2D(512, kernel_size=(3, 3), activation='relu')(maxp_4)
maxp_5 = MaxPooling2D(pool_size=(2, 2))(conv_5)
flatten = Flatten()(maxp_5)

# Age branch
age_dense_1 = Dense(256, activation='relu')(flatten)
age_dense_2 = Dense(128, activation='relu')(age_dense_1)
age_out = Dense(1, activation='linear', name='age_out')(age_dense_2)

# Gender branch
gender_dense_1 = Dense(256, activation='relu')(flatten)
gender_dense_2 = Dense(128, activation='relu')(gender_dense_1)
gender_out = Dense(1, activation='sigmoid', name='gender_out')(gender_dense_2)


model = Model(inputs=inputs, outputs=[gender_out, age_out])  # defining model inputs and outputs

model.compile(loss=['binary_crossentropy', 'mean_squared_error'],
              optimizer=Adam(lr=1e-4),
              metrics={'gender_out': 'accuracy', 'age_out': r2_score})

model.summary()

In [ ]:
plot_model(model,to_file='model_plot.png', show_shapes=True) #plotting model structure

This modeling system is designed to simultaneously predict sex and age from facial images. It starts with an input image of 128x128 pixels, then uses a series of convolutional layers followed by an activation function to capture the image features. Maximum pooling levels help reduce data reduction while maintaining the required pattern. The shared features fall into two branches: one predicts age using two complex layers and linear activation, the other predicts sex using the same layers with sigmoid activation These branches work together in the model, guided by loss activities that help them learn. The result is a versatile system that can handle both classification (sex) and regression (age) tasks simultaneously

In [ ]:
# Learning Rate Reducer
learn_control = ReduceLROnPlateau(monitor='val_accuracy', patience=5,
                                  verbose=1,factor=0.2, min_lr=1e-4)

In [ ]:
history = model.fit(
    x=X_train,  # training inputs
    y=[y_train, y2_train],  # training outputs
    validation_data=(X_test, [y_test, y2_test]),  # testing data
    batch_size=32,
    epochs=20,
    callbacks=[learn_control]
)

In [ ]:
# Create subplots
fig, axs = plt.subplots(2, 2, figsize=(8, 8))  # 3 rows, 2 columns

# Plot loss
axs[0, 0].plot(history.history['loss'])
axs[0, 0].plot(history.history['val_loss'])
axs[0, 0].set_title('Model Loss')
axs[0, 0].set_xlabel('Epochs')
axs[0, 0].set_ylabel('Loss')
axs[0, 0].legend(['Train', 'Validation'])

# Plot gender accuracy
axs[0, 1].plot(history.history['gender_out_accuracy'])
axs[0, 1].plot(history.history['val_gender_out_accuracy'])
axs[0, 1].set_title('Gender Accuracy')
axs[0, 1].set_xlabel('Epochs')
axs[0, 1].set_ylabel('Accuracy')
axs[0, 1].legend(['Train', 'Validation'])

# Plot age R^2 score
axs[1, 0].plot(history.history['age_out_r2_score'])
axs[1, 0].plot(history.history['val_age_out_r2_score'])
axs[1, 0].set_title('Age R^2 Score')
axs[1, 0].set_xlabel('Epochs')
axs[1, 0].set_ylabel('R^2 Score')
axs[1, 0].legend(['Train', 'Validation'])

# Plot age loss
axs[1, 1].plot(history.history['age_out_loss'])
axs[1, 1].plot(history.history['val_age_out_loss'])
axs[1, 1].set_title('Age Loss')
axs[1, 1].set_xlabel('Epochs')
axs[1, 1].set_ylabel('Loss')
axs[1, 1].legend(['Train', 'Validation'])

# Adjust spacing between subplots
plt.tight_layout()

# Show the plots
plt.show()


## Model Evaluation

In [ ]:
model_json = model.to_json()
with open("model.json", "w") as json_file:
    json_file.write(model_json)
model.save('model.h5')

In [ ]:
# Evaluate the model on test data for gender and age
gender_pred, age_pred = model.predict(X_test)

In [ ]:
gender_preds = (gender_pred > 0.5).astype(int) #rounding of the predicted value

In [ ]:
# Generate classification report with class names
gender_class_names = ['Female', 'Male']  # Add the gender class names
classification_rep = classification_report(y_test, gender_preds, target_names=gender_class_names)
print("Gender Classification Report:")
print(classification_rep)

In [ ]:
# Plot confusion matrix heatmap with class names

confusion_matrix = confusion_matrix(y_test,gender_preds)
gender_names = ['Female', 'Male']  # Add the gender class names
sns.heatmap(confusion_matrix, annot=True, cmap='gist_ncar', fmt='g', xticklabels=gender_names, yticklabels=gender_names)

plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')

In [ ]:
# predicted probabilities for the 'Male' class
fpr, tpr, _ = roc_curve(y_test, gender_pred)  # Using probabilities for the 'Male' class
roc_auc = auc(fpr, tpr)

# Plotting the ROC curve
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic')
plt.legend(loc="lower right")
plt.show()

print("AUC:", roc_auc)

Analysis of the gender classification function of our model reveals promising results. With an overall accuracy of 85%, the model exhibits balanced performance in predicting both sexes. It demonstrates greater accuracy in predicting females (82%) compared to males (89%), while maintaining a respectable recall rate of 91% for females and 79% for males This is consistent with the average rate, with a balance of precision and recall at 85 %. The confusion matrix visually represents the predictions of the model, and demonstrates its ability to correctly classify a large proportion of women and men while not under-classifying Furthermore, an impressive AUC score of 0.93 for the ROC the curve highlights the strong ability of the model to predict gender differences was predicted to be successful.

In [ ]:
# Calculate MSE
age_mse = mean_squared_error(y2_test, age_pred)

# Calculate MAE
age_mae = mean_absolute_error(y2_test, age_pred)

# Calculate R^2 score
age_r2 = r2_score(y2_test, age_pred)

# Print the evaluation metrics
print("Age Prediction Evaluation Metrics:")
print("MSE:", age_mse)
print("MAE:", age_mae)
print("R^2 Score:", age_r2)

In [ ]:
# Plot real and predicted age values
plt.figure(figsize=(8, 6))
plt.scatter(y2_test, age_pred)
plt.plot([np.min(y2_test), np.max(y2_test)], [np.min(y2_test), np.max(y2_test)], 'k--')
plt.xlabel('Real Age')
plt.ylabel('Predicted Age')
plt.title('Age Prediction')
plt.legend(['Ideal Line', 'Predicted'])
plt.show()

Follwoing are the metrics given by age model:

**1-Mean Squared Error (MSE): 78.03**

**2-Mean Absolute Error (MAE): 6.14**

**3-R-squared Score: 0.80**

These metrics together signify the model's efficacy in estimating ages from facial expressions. The relatively low MSE and MAE values emphasize accurate age predictions, while the R-squared score indicates that around 80% of the age variance is explained by the model.

As we have done looking at model performance, now we wil take some random images from dataset and make prediction on that and see what are real and prediction our model given.

In [ ]:
## Select 5 random images from the test data
num_images = 5
random_indices = np.random.choice(len(X_test), num_images, replace=False)
selected_images = X_test[random_indices]
selected_gender_labels = y_test[random_indices]
selected_age_labels = y2_test[random_indices]

# Make predictions for selected images
selected_gender_preds, selected_age_preds = model.predict(selected_images)

# Convert gender predictions to class labels
selected_gender_preds_classes = (selected_gender_preds > 0.5).astype(int)


# Display the images with their predicted and real age and gender
fig, axs = plt.subplots(num_images, 1, figsize=(8, 12))

for i in range(num_images):
    # Display the image
    axs[i].imshow(selected_images[i].squeeze(), cmap='gray')
    axs[i].axis('off')

    # Construct the label string
    real_label = f"Real - Gender: {selected_gender_labels[i]}, Age: {selected_age_labels[i]}"
    pred_label = f"Predicted - Gender: {selected_gender_preds_classes[i].item()}, Age: {selected_age_preds[i].item():.1f}"

    # Display the labels
    axs[i].text(0, -10, real_label, fontsize=12, color='red')
    axs[i].text(0, selected_images[i].shape[0] + 10, pred_label, fontsize=12, color='green')

plt.tight_layout()
plt.show()

# Conclusion

Our model did a really good job at telling whether someone is a guy or a girl, with accuracy around 83%. This means it's quite reliable for figuring out gender in real-life situations. When it comes to guessing someone's age, our model's results look promising too. The mistakes it made were not too far off, and it got pretty close to the actual ages.

Looking ahead, there are exciting ways to make our model even better. For the gender part, we could focus on tuning it to understand specific things that make guys and girls look different. We also need to be careful about any biases the model might have and fix them.

For age prediction, we could try fancier model designs and add in more details that help guess ages better. With these tweaks, we could make our predictions even more accurate.

We went a step further and turned it into a website using Flask. This means anyone can use it online to get instant predictions. It's a user-friendly way to share our model's abilities with more people.

In the end, our model's success in gender and age prediction tasks shows its potential for practical use. As we fine-tune and explore more, we could apply it in different areas. And making it available as a web app proves that our project is heading towards real-world impact.